# Notebook 15: Federated Learning Basics — FedAvg and Communication Rounds

## Why This Matters

The canonical ML training pipeline — collect all data in one place, train centrally — is increasingly impractical. Medical institutions can't share patient records. Banks can't share transaction data. Phones can't upload all their sensor readings to a server. The data needed to train the best models is legally, ethically, or practically impossible to centralize.

Federated learning (FL), introduced by McMahan et al. at Google in 2017, inverts the training paradigm: instead of bringing data to the model, you bring the model to the data. Each client trains on its local data and sends only model updates (gradients or weights) to a central server, which aggregates them. Raw data never leaves the device.

Google uses FL in production for Gboard (keyboard next-word prediction), Google Assistant, and Chrome. Apple uses it for Siri and QuickType. The EU's GDPR has accelerated FL adoption by making raw data aggregation risky — FL offers a practical path to training on distributed sensitive data without violating data residency requirements.

## Background

### FedAvg — The Founding Algorithm

McMahan et al.'s paper *"Communication-Efficient Learning of Deep Networks from Decentralized Data"* (2017) introduced FedAvg. The key insight: instead of sharing gradients after every mini-batch (like distributed SGD), each client runs *multiple local epochs* of SGD before communicating. This dramatically reduces communication rounds — the dominant bottleneck in mobile FL where clients have limited bandwidth.

The FedAvg algorithm:
1. **Server initializes** global model weights `w_0`
2. **Server broadcasts** current weights to a random subset of clients (e.g., 10 per round)
3. **Each client** runs `E` local epochs of SGD on its private data, producing updated weights `w_i`
4. **Clients upload** their updated weights (or the delta `Δw_i = w_i - w_0`)
5. **Server aggregates** via weighted average: `w_{t+1} = Σ (n_i / N) * w_i` where n_i is client i's data size
6. Repeat for `T` communication rounds

The weighted average is crucial: clients with more data contribute proportionally more to the global model.

### The Non-IID Problem

The biggest challenge in FL is *data heterogeneity*. In centralized training, we assume IID (independent identically distributed) samples. In FL, each client has data from its own distribution — a phone user's typing habits differ dramatically from another user's. Federated data is fundamentally non-IID.

Non-IID data causes FedAvg to converge slower and to a worse solution than centralized training. After local training, each client's weights drift toward their local optimum — further away from the global optimum. When the server averages these drifted weights, it gets something noisier than the true global gradient. This *client drift* is one of the central research problems in FL.

### IID vs. Non-IID Data Distribution

We model non-IID distribution using *label skew*: each client only has data from a subset of classes. The extreme case is 1 class per client; a more realistic case is 2-3 classes per client with some overlap. Non-IID severity is controlled by the Dirichlet distribution parameter α: small α (e.g., 0.1) → extreme non-IID (each client mostly one class); large α (e.g., 10) → nearly IID.

### Communication Efficiency

Communicating full model weights every round is expensive — a large model might be hundreds of MB. Several techniques reduce communication cost:
- **Gradient compression**: send only top-k gradients by magnitude (sparsification)
- **Quantization**: send 8-bit or 4-bit weights instead of float32
- **FedProx**: add a proximal term to each client's local objective to prevent drift
- **SCAFFOLD**: use control variates to correct for client drift without extra communication

### FL + Differential Privacy

Even though raw data never leaves the device, the model updates themselves can leak information (as we'll see in notebook 16 on gradient inversion attacks). Combining FL with DP (specifically, client-level DP applied to the uploaded deltas) provides formal privacy guarantees. Google's production FL system uses DP with secure aggregation.

## Structure of This Notebook

1. Set up a simulated FL environment with multiple clients
2. Implement IID and non-IID data partitioning (Dirichlet)
3. Implement FedAvg with weighted aggregation
4. Compare convergence: IID vs. non-IID, more vs. fewer local epochs
5. Implement FedProx to mitigate client drift
6. Communication efficiency: gradient compression

## What You'll Learn

- How FedAvg's weighted aggregation works and why data size matters
- Why non-IID data slows convergence and causes client drift
- The role of local epochs (E) in the communication-accuracy tradeoff
- How FedProx's proximal term corrects for drift
- How gradient sparsification reduces communication cost

In [ ]:
%pip install torch torchvision numpy matplotlib --quiet

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset, TensorDataset
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
import copy
from collections import defaultdict

device = (
    torch.device('mps') if torch.backends.mps.is_available()
    else torch.device('cuda') if torch.cuda.is_available()
    else torch.device('cpu')
)
print(f'Device: {device}')
torch.manual_seed(42)
np.random.seed(42)

## 1. Data and Model Setup

In [ ]:
class MnistCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.drop = nn.Dropout(0.25)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 64 * 7 * 7)
        x = F.relu(self.fc1(x))
        x = self.drop(x)
        return self.fc2(x)


transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
train_dataset = torchvision.datasets.MNIST('./data', train=True, download=True, transform=transform)
test_dataset = torchvision.datasets.MNIST('./data', train=False, download=True, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=512, shuffle=False)

# Extract labels for partitioning
train_labels = np.array([train_dataset[i][1] for i in range(len(train_dataset))])
print(f'Train samples: {len(train_dataset)}, label distribution: {np.bincount(train_labels)}')

## 2. Data Partitioning — IID vs. Non-IID (Dirichlet)

In [ ]:
def iid_partition(dataset, n_clients):
    """
    IID partition: randomly shuffle and split equally.
    Each client gets a representative sample of all classes.
    """
    all_idx = np.random.permutation(len(dataset))
    splits = np.array_split(all_idx, n_clients)
    return [list(s) for s in splits]


def dirichlet_partition(labels, n_clients, alpha=0.5):
    """
    Non-IID partition using Dirichlet distribution.
    alpha controls heterogeneity:
    - Small alpha (0.1): extreme skew — each client mostly one class
    - Large alpha (10): near-IID — each client has all classes
    """
    n_classes = len(np.unique(labels))
    client_indices = [[] for _ in range(n_clients)]

    for c in range(n_classes):
        class_idx = np.where(labels == c)[0]
        np.random.shuffle(class_idx)

        # Dirichlet proportions for this class across clients
        proportions = np.random.dirichlet(np.repeat(alpha, n_clients))
        # Convert to counts, ensuring at least 1 sample per client
        proportions = np.maximum(proportions, 1 / len(class_idx))
        proportions = proportions / proportions.sum()
        counts = (proportions * len(class_idx)).astype(int)
        counts[-1] = len(class_idx) - counts[:-1].sum()  # fix rounding

        start = 0
        for i, count in enumerate(counts):
            client_indices[i].extend(class_idx[start:start+count])
            start += count

    return client_indices


def visualize_partitions(client_indices, labels, title, n_clients=10):
    """Show class distribution per client as a stacked bar chart."""
    n_classes = 10
    dist = np.zeros((n_clients, n_classes))
    for i, idx in enumerate(client_indices[:n_clients]):
        client_labels = labels[idx]
        for c in range(n_classes):
            dist[i, c] = (client_labels == c).sum()

    # Normalize to proportions
    dist_norm = dist / (dist.sum(axis=1, keepdims=True) + 1e-8)

    fig, ax = plt.subplots(figsize=(12, 3))
    bottom = np.zeros(n_clients)
    colors = plt.cm.tab10(np.linspace(0, 1, n_classes))
    for c in range(n_classes):
        ax.bar(range(n_clients), dist_norm[:, c], bottom=bottom, color=colors[c], label=str(c))
        bottom += dist_norm[:, c]
    ax.set_xlabel('Client')
    ax.set_ylabel('Class Proportion')
    ax.set_title(title)
    ax.legend(title='Digit', bbox_to_anchor=(1.01, 1), loc='upper left', ncol=2, fontsize=8)
    plt.tight_layout()
    plt.show()


N_CLIENTS = 10

iid_indices = iid_partition(train_dataset, N_CLIENTS)
noniid_indices_mild = dirichlet_partition(train_labels, N_CLIENTS, alpha=0.5)
noniid_indices_extreme = dirichlet_partition(train_labels, N_CLIENTS, alpha=0.1)

visualize_partitions(iid_indices, train_labels, 'IID Partition — Each client sees all classes uniformly')
visualize_partitions(noniid_indices_mild, train_labels, 'Non-IID (α=0.5) — Mild label skew')
visualize_partitions(noniid_indices_extreme, train_labels, 'Non-IID (α=0.1) — Extreme label skew')

print(f'IID: client 0 has {len(iid_indices[0])} samples')
print(f'Non-IID: client sizes range {min(len(x) for x in noniid_indices_extreme)}'
      f'–{max(len(x) for x in noniid_indices_extreme)}')

## 3. FedAvg Implementation

In [ ]:
def get_model_weights(model):
    """Extract model weights as a dict of tensors."""
    return copy.deepcopy(model.state_dict())


def set_model_weights(model, weights):
    """Load weights into model."""
    model.load_state_dict(weights)


def federated_average(client_weights, client_sizes):
    """
    Weighted average of client model weights.
    Weight for client i = n_i / N (proportional to data size).
    """
    total_size = sum(client_sizes)
    avg_weights = {}

    for key in client_weights[0].keys():
        weighted_sum = sum(
            w[key].float() * (sz / total_size)
            for w, sz in zip(client_weights, client_sizes)
        )
        avg_weights[key] = weighted_sum

    return avg_weights


def client_update(
    global_weights,
    client_data_idx,
    dataset,
    local_epochs: int,
    lr: float,
    batch_size: int = 32,
    mu: float = 0.0,  # FedProx proximal term (0 = standard FedAvg)
):
    """
    Client local training: runs E epochs of SGD on local data.

    mu > 0 enables FedProx: adds proximal term ||w - w_global||² to loss.
    This prevents client weights from drifting too far from the global model.
    """
    model = MnistCNN().to(device)
    set_model_weights(model, global_weights)

    global_params = {name: p.clone().detach() for name, p in model.named_parameters()}

    optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    client_loader = DataLoader(
        Subset(dataset, client_data_idx),
        batch_size=batch_size, shuffle=True, drop_last=False
    )

    model.train()
    for _ in range(local_epochs):
        for x, y in client_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            loss = F.cross_entropy(model(x), y)

            # FedProx proximal term: penalize distance from global weights
            if mu > 0:
                prox = sum(
                    ((p - global_params[name].to(device)) ** 2).sum()
                    for name, p in model.named_parameters()
                )
                loss = loss + (mu / 2) * prox

            loss.backward()
            optimizer.step()

    return get_model_weights(model), len(client_data_idx)


def evaluate_global(global_weights, test_loader):
    model = MnistCNN().to(device)
    set_model_weights(model, global_weights)
    model.eval()
    correct = 0
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            correct += (model(x).argmax(1) == y).sum().item()
    return correct / len(test_loader.dataset)


def fedavg(
    client_indices,
    dataset,
    test_loader,
    n_rounds: int = 20,
    clients_per_round: int = 5,
    local_epochs: int = 3,
    lr: float = 0.01,
    mu: float = 0.0,
    label: str = 'FedAvg',
):
    """Full FedAvg training loop."""
    global_model = MnistCNN().to(device)
    global_weights = get_model_weights(global_model)
    history = []

    n_clients = len(client_indices)

    for round_idx in range(n_rounds):
        # Select random subset of clients
        selected = np.random.choice(n_clients, clients_per_round, replace=False)

        # Client updates
        client_ws = []
        client_szs = []
        for c in selected:
            w, sz = client_update(
                global_weights, client_indices[c], dataset,
                local_epochs=local_epochs, lr=lr, mu=mu
            )
            client_ws.append(w)
            client_szs.append(sz)

        # Aggregate
        global_weights = federated_average(client_ws, client_szs)

        # Evaluate
        acc = evaluate_global(global_weights, test_loader)
        history.append(acc)

        if (round_idx + 1) % 5 == 0 or round_idx == 0:
            print(f'[{label}] Round {round_idx+1:2d}/{n_rounds}: acc={acc:.4f}')

    return history


print('FedAvg implemented. Running experiments...')

## 4. IID vs. Non-IID Convergence Comparison

In [ ]:
N_ROUNDS = 20
CLIENTS_PER_ROUND = 5

print('--- IID partition ---')
hist_iid = fedavg(iid_indices, train_dataset, test_loader,
                  n_rounds=N_ROUNDS, clients_per_round=CLIENTS_PER_ROUND,
                  local_epochs=3, lr=0.01, label='IID')

print('\n--- Non-IID partition (α=0.5) ---')
hist_noniid = fedavg(noniid_indices_mild, train_dataset, test_loader,
                     n_rounds=N_ROUNDS, clients_per_round=CLIENTS_PER_ROUND,
                     local_epochs=3, lr=0.01, label='Non-IID (α=0.5)')

print('\n--- Non-IID extreme (α=0.1) ---')
hist_noniid_ext = fedavg(noniid_indices_extreme, train_dataset, test_loader,
                         n_rounds=N_ROUNDS, clients_per_round=CLIENTS_PER_ROUND,
                         local_epochs=3, lr=0.01, label='Non-IID (α=0.1)')

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(range(1, N_ROUNDS+1), hist_iid, 'b-o', markersize=4, label='IID')
ax.plot(range(1, N_ROUNDS+1), hist_noniid, 'orange', marker='s', markersize=4, label='Non-IID (α=0.5)')
ax.plot(range(1, N_ROUNDS+1), hist_noniid_ext, 'r-^', markersize=4, label='Non-IID (α=0.1, extreme)')
ax.set_xlabel('Communication Rounds')
ax.set_ylabel('Global Model Test Accuracy')
ax.set_title('FedAvg Convergence: IID vs. Non-IID Data')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Final accuracy — IID: {hist_iid[-1]:.4f}, '
      f'Non-IID mild: {hist_noniid[-1]:.4f}, '
      f'Non-IID extreme: {hist_noniid_ext[-1]:.4f}')

## 5. Effect of Local Epochs on Communication vs. Accuracy

In [ ]:
local_epoch_configs = [1, 3, 5, 10]
epoch_histories = {}

for E in local_epoch_configs:
    print(f'\n--- Local epochs E={E} (Non-IID α=0.5) ---')
    hist = fedavg(noniid_indices_mild, train_dataset, test_loader,
                  n_rounds=15, clients_per_round=5,
                  local_epochs=E, lr=0.01, label=f'E={E}')
    epoch_histories[E] = hist

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Accuracy vs. rounds
for E, hist in epoch_histories.items():
    axes[0].plot(range(1, len(hist)+1), hist, marker='o', markersize=3, label=f'E={E}')
axes[0].set_xlabel('Communication Rounds')
axes[0].set_ylabel('Test Accuracy')
axes[0].set_title('Accuracy per Communication Round')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Final accuracy vs. total gradient steps (rounds * E)
total_steps = {E: np.arange(1, len(h)+1) * E * 5 for E, h in epoch_histories.items()}
for E, hist in epoch_histories.items():
    steps = total_steps[E]
    axes[1].plot(steps, hist, marker='o', markersize=3, label=f'E={E}')
axes[1].set_xlabel('Total Gradient Steps (rounds × E × clients)')
axes[1].set_ylabel('Test Accuracy')
axes[1].set_title('Accuracy per Computation Budget')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Effect of Local Epochs on Communication-Efficiency Tradeoff')
plt.tight_layout()
plt.show()

print('Key insight: More local epochs (E) → fewer rounds needed, but client drift increases.')
print('Under non-IID data, E>5 often hurts because drift dominates the averaging step.')

## 6. FedProx — Mitigating Client Drift

In [ ]:
print('--- FedAvg (μ=0) on extreme non-IID ---')
hist_fedavg_ext = fedavg(
    noniid_indices_extreme, train_dataset, test_loader,
    n_rounds=20, clients_per_round=5, local_epochs=5, lr=0.01, mu=0.0,
    label='FedAvg (μ=0)'
)

print('\n--- FedProx (μ=0.1) on extreme non-IID ---')
hist_fedprox = fedavg(
    noniid_indices_extreme, train_dataset, test_loader,
    n_rounds=20, clients_per_round=5, local_epochs=5, lr=0.01, mu=0.1,
    label='FedProx (μ=0.1)'
)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(range(1, 21), hist_fedavg_ext, 'r-o', markersize=4, label='FedAvg (no drift correction)')
ax.plot(range(1, 21), hist_fedprox, 'b-s', markersize=4, label='FedProx (μ=0.1)')
ax.set_xlabel('Communication Rounds')
ax.set_ylabel('Test Accuracy')
ax.set_title('FedAvg vs. FedProx on Extreme Non-IID Data (α=0.1)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'FedAvg final acc: {hist_fedavg_ext[-1]:.4f}')
print(f'FedProx final acc: {hist_fedprox[-1]:.4f}')
print(f'FedProx improvement: {(hist_fedprox[-1] - hist_fedavg_ext[-1])*100:.2f}%')

## 7. Gradient Compression — Communication Efficiency

In [ ]:
def compute_weight_delta(global_weights, client_weights):
    """Compute weight delta (what client would upload)."""
    delta = {}
    for key in global_weights:
        delta[key] = client_weights[key] - global_weights[key]
    return delta


def sparsify_delta(delta, sparsity=0.99):
    """
    Top-k sparsification: keep only top (1-sparsity) fraction of
    weight deltas by magnitude. Zeros the rest.
    Reduces communication by sparsity fraction.
    """
    all_vals = torch.cat([v.flatten().abs() for v in delta.values()])
    threshold = torch.quantile(all_vals, sparsity)

    sparse_delta = {}
    for key, v in delta.items():
        mask = v.abs() >= threshold
        sparse_delta[key] = v * mask

    # Measure actual sparsity
    total = sum(v.numel() for v in sparse_delta.values())
    nonzero = sum((v != 0).sum().item() for v in sparse_delta.values())

    return sparse_delta, nonzero / total


def apply_delta(global_weights, delta):
    """Apply delta to global weights to get updated weights."""
    updated = {}
    for key in global_weights:
        updated[key] = global_weights[key] + delta.get(key, torch.zeros_like(global_weights[key]))
    return updated


# Analyze compression: how much can we sparsify without losing accuracy?
global_model = MnistCNN().to(device)
global_weights_test = get_model_weights(global_model)

# Perform one client update to get a realistic delta
client_w, _ = client_update(global_weights_test, iid_indices[0], train_dataset, local_epochs=3, lr=0.01)
delta = compute_weight_delta(global_weights_test, client_w)

total_params = sum(v.numel() for v in delta.values())
print(f'Total parameters in delta: {total_params:,}')
print(f'Full delta size: {total_params * 4 / 1024:.1f} KB (float32)')
print()

sparsity_levels = [0.0, 0.5, 0.9, 0.95, 0.99]
print(f'{"Sparsity":>10} {"Transmitted":>12} {"Compression":>12} {"KB transmitted":>15}')
print('-' * 55)

for sp in sparsity_levels:
    sparse_d, actual_density = sparsify_delta(delta, sp)
    nonzero = sum((v != 0).sum().item() for v in sparse_d.values())
    compression = total_params / (nonzero + 1)
    kb = nonzero * 4 / 1024
    print(f'{sp:>10.0%} {nonzero:>12,} {compression:>12.1f}x {kb:>15.1f}')

print()
print('In practice, 90-99% sparsification with error feedback (accumulating residuals)')
print('loses only 1-3% accuracy vs. full gradient communication.')

## Summary

### What We Built

| Component | Description | Key Detail |
|-----------|-------------|------------|
| `iid_partition()` | Random uniform split | Baseline — each client representative |
| `dirichlet_partition()` | Label-skewed split via Dirichlet(α) | α=0.1 extreme, α=10 near-IID |
| `federated_average()` | Weighted aggregation `Σ (n_i/N) w_i` | Larger clients contribute more |
| `client_update()` | Local E epochs of SGD | Optional FedProx proximal term |
| `fedavg()` | Full training loop | T rounds × C clients per round |
| FedProx (μ>0) | Proximal regularization | Reduces drift on non-IID data |
| `sparsify_delta()` | Top-k sparsification | 99% sparsity = 100x compression |

### Key Takeaways

- **FedAvg works well on IID data** and converges in fewer rounds than centralized SGD because local epochs amortize communication cost.
- **Non-IID data is FL's core challenge.** Extreme label skew (α=0.1) degrades convergence significantly — clients pull the global model toward their local optima.
- **More local epochs is not always better.** Under non-IID data, E>5 often hurts due to client drift dominating the averaging step.
- **FedProx's proximal term prevents drift** by penalizing deviation from the global model. It's a simple but effective regularizer for heterogeneous FL.
- **Gradient compression is highly effective.** 99% sparsification (100x compression) with top-k selection loses very little accuracy when combined with error feedback.

### FL in Practice

Google's production FL system (described in their 2019 paper) uses:
- Tens of thousands of clients per round (from millions of devices)
- Secure aggregation (clients' updates are encrypted before the server sees them)
- Differential privacy on the aggregated update
- Adaptive clipping thresholds that update automatically

The gap between this notebook's simulation and production FL is significant — but the core FedAvg algorithm and the IID/non-IID tradeoff remain central.

Next: Notebook 16 covers federated learning attacks — Byzantine clients, gradient inversion on uploaded updates, and poisoning in the FL setting.